In [ ]:
# ============================================================
# LABORATOR U10 - SERIE LUNARA: MEDII MOBILE, RATE, INDICI, VALORI REALE
# Stil: parametri la inceput, pasi numerotati, variabile in romana (fara diacritice)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# CARTEA DE IDENTITATE A INDICATORULUI (parametri, nu cifre in cod)
# ------------------------------------------------------------
PARAMETRI_INDICATOR = {
    'fisier_date':        None,          # ex: "date_u10.csv"; None => folosim datele demo
    'variabila':          'indicele preturilor de consum (IPC)',
    'unitate':            'procente (indice, an baza = 100)',
    'periodicitate':      'lunar',
    'an_baza':            2023,          # media anului de baza = 100
    'deflator':           'IPC (indicele preturilor de consum)',
    'metoda':             'medie mobila 12 luni, rate in lant si anuale, indici cu baza fixa',
    'rata_comunicata':    'rata anuala a inflatiei (%)',
    'lungime_medie_mobila': 12,          # n = 12 (pentru date trimestriale: n = 4)
    'verificari_fisa': {                 # valorile din fisa ta (completeaza!)
        # 'iunie 2024':  {'medie_mobila': ..., 'rata_lunara': ..., 'rata_anuala': ...},
        # 'iunie 2025':  {...},
        # 'decembrie 2025': {...},
    },
}

# ------------------------------------------------------------
# Helper: mapare etichete romanesti -> Period (fara parsare de string)
# ------------------------------------------------------------
LUNI_RO = {
    'ianuarie': 1, 'februarie': 2, 'martie': 3, 'aprilie': 4,
    'mai': 5, 'iunie': 6, 'iulie': 7, 'august': 8,
    'septembrie': 9, 'octombrie': 10, 'noiembrie': 11, 'decembrie': 12,
}

def eticheta_la_period(eticheta: str) -> pd.Period:
    """'iunie 2024' -> Period('2024-06', 'M'). Nu depinde de locale."""
    parti = eticheta.lower().strip().split()
    if len(parti) != 2 or parti[0] not in LUNI_RO:
        raise ValueError(f"Eticheta necunoscuta: {eticheta!r}")
    luna_nr = LUNI_RO[parti[0]]
    an = int(parti[1])
    return pd.Period(f"{an}-{luna_nr:02d}", freq='M')

# ------------------------------------------------------------
# DATE DEMO: 36 luni (2023-01 ... 2025-12), valoare nominala + indice preturi
# Inlocuieste cu: pd.read_csv(PARAMETRI_INDICATOR['fisier_date'])
# ------------------------------------------------------------
luni = pd.period_range('2023-01', '2025-12', freq='M')
np.random.seed(7)
trend = np.linspace(100, 130, len(luni))
sezon = 5 * np.sin(2 * np.pi * np.arange(len(luni)) / 12)
zgomot = np.random.normal(0, 1.5, len(luni))
valoare = np.round(trend + sezon + zgomot, 1)
indice_preturi = np.round(np.linspace(100, 112, len(luni)) + np.random.normal(0, 0.3, len(luni)), 1)

date_brute_u10 = pd.DataFrame({
    'luna':           luni.astype(str),
    'an':             luni.year,
    'luna_nr':        luni.month,
    'valoare':        valoare,
    'indice_preturi': indice_preturi,
})

# ============================================================
# PASUL 0 - Citirea / pregatirea datelor
# ============================================================
if PARAMETRI_INDICATOR['fisier_date']:
    cadru_date_u10 = pd.read_csv(PARAMETRI_INDICATOR['fisier_date'])
else:
    cadru_date_u10 = date_brute_u10.copy()

cadru_date_u10['luna'] = pd.PeriodIndex(cadru_date_u10['luna'], freq='M')
cadru_date_u10 = cadru_date_u10.sort_values('luna').reset_index(drop=True)
for col in ['valoare', 'indice_preturi']:
    cadru_date_u10[col] = pd.to_numeric(cadru_date_u10[col], errors='coerce')

print("=" * 60)
print("PAS 0 - Date incarcate")
print("=" * 60)
print(cadru_date_u10.head(5).to_string(index=False))
print(f"Total luni: {len(cadru_date_u10)}")

# ============================================================
# PASUL 1 - Media fiecarui an + luna maxima/minima din fiecare an
# ============================================================
media_anuala = cadru_date_u10.groupby('an')['valoare'].mean().round(2)
extrem_max = (cadru_date_u10.loc[cadru_date_u10.groupby('an')['valoare'].idxmax(),
                                 ['an', 'luna', 'valoare']]
              .rename(columns={'luna': 'luna_max', 'valoare': 'val_max'}))
extrem_min = (cadru_date_u10.loc[cadru_date_u10.groupby('an')['valoare'].idxmin(),
                                  ['an', 'luna', 'valoare']]
              .rename(columns={'luna': 'luna_min', 'valoare': 'val_min'}))
tabel_anual = (extrem_max.merge(extrem_min, on='an')
               .assign(media_anuala=media_anuala.values))

print("\n" + "=" * 60)
print("PASUL 1 - Media anuala, luna max/min")
print("=" * 60)
print(tabel_anual.to_string(index=False))

# ============================================================
# PASUL 2 - Media mobila pe 12 luni + verificare fereastra
# ============================================================
n_mm = PARAMETRI_INDICATOR['lungime_medie_mobila']
cadru_date_u10['medie_mobila'] = (
    cadru_date_u10['valoare'].rolling(window=n_mm, min_periods=n_mm).mean().round(2)
)

# Regula ferestrei: suma noua = suma veche - valoarea cea mai veche + valoarea noua
serie = cadru_date_u10['valoare'].to_numpy()
verificari_fereastra = []
for i in range(n_mm, len(serie)):
    suma_veche = serie[i - n_mm:i].sum()
    suma_noua  = serie[i - n_mm + 1:i + 1].sum()
    ok = np.isclose(suma_noua, suma_veche - serie[i - n_mm] + serie[i])
    verificari_fereastra.append(ok)
print("\n" + "=" * 60)
print("PASUL 2 - Media mobila 12 luni")
print("=" * 60)
print(f"Ferestre verificate: {len(verificari_fereastra)} | toate corecte: {all(verificari_fereastra)}")
print(cadru_date_u10.loc[cadru_date_u10['medie_mobila'].notna(),
                        ['luna', 'valoare', 'medie_mobila']].tail(5).to_string(index=False))

# Verificare cu fisa (daca ai completat VERIFICARI_FISA)
for eticheta, valori in PARAMETRI_INDICATOR['verificari_fisa'].items():
    if 'medie_mobila' in valori:
        luna_cautata = eticheta_la_period(eticheta)
        gasit = cadru_date_u10.loc[cadru_date_u10['luna'] == luna_cautata, 'medie_mobila']
        if not gasit.empty:
            print(f"  {eticheta}: program={gasit.iloc[0]:.2f} | fisa={valori['medie_mobila']:.2f} "
                  f"| diff={gasit.iloc[0]-valori['medie_mobila']:+.2f}")

# Grafic serie + medie mobila
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cadru_date_u10['luna'].astype(str), cadru_date_u10['valoare'], marker='o', label='Serie')
ax.plot(cadru_date_u10['luna'].astype(str), cadru_date_u10['medie_mobila'], label=f'MM {n_mm} luni')
ax.set_title('Serie lunara si medie mobila'); ax.legend(); ax.tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.savefig('grafic_u10.png', dpi=100); plt.close()
print("Grafic salvat: grafic_u10.png")

# ============================================================
# PASUL 3 - Rate: lunara (fata de luna precedenta) si anuala
# ============================================================
cadru_date_u10['rata_lunara'] = cadru_date_u10['valoare'].pct_change() * 100
cadru_date_u10['rata_anuala'] = cadru_date_u10['valoare'].pct_change(n_mm) * 100

dif_rate = (cadru_date_u10['rata_anuala'] - cadru_date_u10['rata_lunara']).abs()
idx_max = dif_rate.idxmax()
print("\n" + "=" * 60)
print("PASUL 3 - Rate lunare si anuale")
print("=" * 60)
print(f"Luna cu cea mai mare diferenta |rata_anuala - rata_lunara|: "
      f"{cadru_date_u10.loc[idx_max, 'luna']} ({dif_rate.iloc[idx_max]:.2f} pp)")

for eticheta, valori in PARAMETRI_INDICATOR['verificari_fisa'].items():
    if 'rata_anuala' in valori:
        luna_cautata = eticheta_la_period(eticheta)
        rand = cadru_date_u10.loc[cadru_date_u10['luna'] == luna_cautata]
        if not rand.empty:
            print(f"  {eticheta}: rata_lunara={rand['rata_lunara'].iloc[0]:.2f}% "
                  f"| rata_anuala={rand['rata_anuala'].iloc[0]:.2f}% "
                  f"| fisa={valori['rata_anuala']:.2f}%")

# ============================================================
# PASUL 4 - Indicii: baza fixa (media 2023 = 100) si indice in lant
# ============================================================
an_baza = PARAMETRI_INDICATOR['an_baza']
media_baza = cadru_date_u10.loc[cadru_date_u10['an'] == an_baza, 'valoare'].mean()
cadru_date_u10['indice_baza_fixa'] = (cadru_date_u10['valoare'] / media_baza * 100).round(2)
cadru_date_u10['indice_lant'] = (cadru_date_u10['valoare'].pct_change() + 1).mul(100).round(2)

print("\n" + "=" * 60)
print("PASUL 4 - Indicii (baza fixa si in lant)")
print("=" * 60)
print(f"Media anului de baza {an_baza}: {media_baza:.2f}")
print(cadru_date_u10[['luna', 'valoare', 'indice_baza_fixa', 'indice_lant']].tail(5).to_string(index=False))

# ============================================================
# PASUL 5 - Valori reale + descompunerea cresterii nominale
# ============================================================
cadru_date_u10['valoare_reala'] = (
    cadru_date_u10['valoare'] / cadru_date_u10['indice_preturi'] * 100
).round(2)
cadru_date_u10['rata_anuala_reala'] = cadru_date_u10['valoare_reala'].pct_change(n_mm) * 100

cadru_date_u10['efect_preturi'] = cadru_date_u10['indice_preturi'].pct_change(n_mm) * 100
cadru_date_u10['efect_volum'] = cadru_date_u10['rata_anuala_reala']
cadru_date_u10['crestere_nominala'] = cadru_date_u10['rata_anuala']

print("\n" + "=" * 60)
print("PASUL 5 - Valori reale si descompunere")
print("=" * 60)
ultimele = cadru_date_u10.dropna(subset=['rata_anuala_reala']).tail(3)
print(ultimele[['luna', 'rata_anuala', 'rata_anuala_reala',
                'efect_preturi', 'efect_volum']].to_string(index=False))

for eticheta, valori in PARAMETRI_INDICATOR['verificari_fisa'].items():
    if 'rata_reala' in valori:
        luna_cautata = eticheta_la_period(eticheta)
        rand = cadru_date_u10.loc[cadru_date_u10['luna'] == luna_cautata]
        if not rand.empty:
            print(f"  {eticheta}: rata_reala={rand['rata_anuala_reala'].iloc[0]:.2f}% "
                  f"| fisa={valori['rata_reala']:.2f}%")

# ============================================================
# PASUL 6 - Raport: indicator - valoare - interpretare (3 cazuri)
# ============================================================
cazuri_raport = []
for eticheta in ['iunie 2024', 'iunie 2025', 'decembrie 2025']:
    luna_cautata = eticheta_la_period(eticheta)
    rand = cadru_date_u10.loc[cadru_date_u10['luna'] == luna_cautata]
    if rand.empty:
        print(f"  [skip] {eticheta}: luna nu exista in serie")
        continue
    r = rand.iloc[0]
    cazuri_raport.append(('Medie mobila 12 luni', eticheta,
                          round(r['medie_mobila'], 2) if pd.notna(r['medie_mobila']) else None,
                          'nivel mediu al ultimelor 12 luni'))
    cazuri_raport.append(('Rata lunara (%)', eticheta, round(r['rata_lunara'], 2),
                          'variatie fata de luna precedenta'))
    cazuri_raport.append(('Rata anuala (%)', eticheta, round(r['rata_anuala'], 2),
                          'variatie fata de aceeasi luna a anului precedent'))
    cazuri_raport.append(('Indice baza fixa', eticheta, round(r['indice_baza_fixa'], 2),
                          f'raportat la media {an_baza} = 100'))
    cazuri_raport.append(('Indice in lant', eticheta, round(r['indice_lant'], 2),
                          'raportat la luna precedenta = 100'))
    cazuri_raport.append(('Rata anuala reala (%)', eticheta,
                          round(r['rata_anuala_reala'], 2) if pd.notna(r['rata_anuala_reala']) else None,
                          'crestere nominala deflatata cu IPC'))

tabel_raport_u10 = pd.DataFrame(cazuri_raport,
                                columns=['Indicator', 'Luna', 'Valoare', 'Interpretare'])

concluzie = ("Concluzie: seria de 36 luni arata o crestere nominala sustinuta, dar dupa "
             "deflatare cu IPC cresterea reala este mai mica; media mobila 12 luni netezeste "
             "sezonalitatea si confirma tendinta de fond.")
tabel_raport_u10.loc[len(tabel_raport_u10)] = ['Concluzie', '', '', concluzie]

print("\n" + "=" * 60)
print("PASUL 6 - Raport final")
print("=" * 60)
print(tabel_raport_u10.to_string(index=False))
tabel_raport_u10.to_csv('raport_u10.csv', index=False)
print("\nRaport salvat in: raport_u10.csv")

# ============================================================
# EXERCITIUL 1 - Date trimestriale (n = 4)
# ============================================================
print("\n" + "=" * 60)
print("EXERCITIUL 1 - Medie mobila pe 4 trimestre")
print("=" * 60)
date_trim = (cadru_date_u10.set_index('luna')
             .resample('Q')
             .agg({'valoare': 'mean'})
             .reset_index())
date_trim['mm4'] = date_trim['valoare'].rolling(4).mean().round(2)
print(date_trim.tail(6).to_string(index=False))

# ============================================================
# EXERCITIUL 2 - Alt an de baza
# ============================================================
print("\n" + "=" * 60)
print("EXERCITIUL 2 - Indice cu baza fixa, alt an de baza")
print("=" * 60)
for an in [2023, 2024]:
    med = cadru_date_u10.loc[cadru_date_u10['an'] == an, 'valoare'].mean()
    idx = (cadru_date_u10['valoare'] / med * 100).round(2)
    print(f"  an_baza={an}: media={med:.2f}, ultimul indice={idx.iloc[-1]:.2f}")

# ============================================================
# EXERCITIUL 3 - Deflator alternativ (indice preturi +2%)
# ============================================================
print("\n" + "=" * 60)
print("EXERCITIUL 3 - Sensibilitate la deflator")
print("=" * 60)
deflator_alt = cadru_date_u10['indice_preturi'] * 1.02
reala_alt = (cadru_date_u10['valoare'] / deflator_alt * 100).round(2)
print(f"  Rata reala finala (deflator original): "
      f"{cadru_date_u10['rata_anuala_reala'].iloc[-1]:.2f}%")
print(f"  Rata reala finala (deflator +2%):     "
      f"{(reala_alt.pct_change(12) * 100).iloc[-1]:.2f}%")

# ============================================================
# EXERCITIUL 4 - Cea mai mare diferenta nominal vs real
# ============================================================
print("\n" + "=" * 60)
print("EXERCITIUL 4 - Diferenta nominal vs real")
print("=" * 60)
diferenta = (cadru_date_u10['rata_anuala'] - cadru_date_u10['rata_anuala_reala']).abs()
idx = diferenta.idxmax()
print(f"  Luna cu diferenta maxima: {cadru_date_u10.loc[idx, 'luna']} "
      f"({diferenta.iloc[idx]:.2f} pp)")

PAS 0 — Date încărcate
   luna   an  luna_nr  valoare  indice_preturi
2023-01 2023        1    102.5            99.6
2023-02 2023        2    102.7           100.8
2023-03 2023        3    106.1           101.2
2023-04 2023        4    108.2           100.9
2023-05 2023        5    106.6           101.6
Total luni: 36

PASUL 1 — Media anuală, luna max/min
  an luna_max  val_max luna_min  val_min  media_anuala
2023  2023-04    108.2  2023-08    100.9        104.72
2024  2024-12    120.3  2024-01    111.0        115.18
2025  2025-05    129.9  2025-01    120.5        124.40

PASUL 2 — Media mobilă 12 luni
Ferestre verificate: 24 | toate corecte: True
   luna  valoare  medie_mobila
2025-08    125.7        121.88
2025-09    120.6        122.32
2025-10    124.1        123.23
2025-11    121.7        123.88
2025-12    126.5        124.40
Grafic salvat: grafic_u10.png

PASUL 3 — Rate lunare și anuale
Luna cu cea mai mare diferență |rata_anuala − rata_lunara|: 2024-08 (14.22 pp)

PASUL 4 — Indic

DateParseError: Unknown datetime string format, unable to parse: IUNIE-2024